# Assistant Axis — Minimal Proof of Concept

**Goal:** Validate the entire pipeline end-to-end in ~10 minutes with minimal data.

**What this does:**
- Uses a **small model** (Gemma-2-2B, ~4GB VRAM — fits on a free Colab T4)
- Generates **10 responses** (5 role-playing + 5 default) instead of 66,000+
- **Skips the LLM judge** — we manually assign scores since we know which are role-playing
- **Skips vLLM** — uses HuggingFace generation (slower but no install headaches)
- Computes the axis and verifies it works via projection

**Requirements:** ~5GB GPU memory, ~10 min runtime

---

## 0. Install Dependencies

Skip vLLM (not needed for POC) to avoid install issues.

In [ ]:
# Uncomment if running on Colab or a fresh environment
# !pip install torch transformers accelerate huggingface_hub scikit-learn numpy tqdm plotly

In [ ]:
import sys
from pathlib import Path

# Add project root to path so we can import assistant_axis
project_root = Path(".").resolve().parent
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

import torch
print(f"PyTorch: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"VRAM: {torch.cuda.get_device_properties(0).total_mem / 1e9:.1f} GB")

---
## 1. Configuration

Everything you might want to change is here.

In [ ]:
# ── Model ──
MODEL_NAME = "google/gemma-2-2b-it"  # ~4GB VRAM, fits on free Colab T4
# MODEL_NAME = "google/gemma-2-9b-it"   # ~18GB VRAM, needs Colab Pro or better
# MODEL_NAME = "google/gemma-2-27b-it"  # ~54GB VRAM, needs A100 80GB

# ── Target layer ──
# Middle layer is a safe default. For known models, get_config() gives the optimal one.
# Gemma-2-2B has 26 layers, so middle = 13
TARGET_LAYER = 13

# ── Role-playing system prompts (our "roles") ──
ROLE_PROMPTS = {
    "pirate": "You are a pirate captain named Blackbeard. You speak in pirate slang, "
              "refer to your crew and ship, and see the world through a pirate's eyes.",
    "doctor": "You are Dr. Sarah Chen, a veteran emergency room physician. You think "
              "in medical terms, reference your clinical experience, and approach "
              "problems with diagnostic reasoning.",
    "poet":   "You are a romantic poet from the 19th century. You speak in flowery, "
              "metaphorical language, reference nature and emotions, and see beauty "
              "in everything.",
}

# ── Default (no persona — just a normal assistant) ──
DEFAULT_PROMPT = None  # No system prompt = default assistant behavior

# ── Questions to ask each role ──
QUESTIONS = [
    "What do you think about when you look at the ocean?",
    "How do you handle a difficult situation?",
    "What advice would you give to someone starting out in life?",
    "Describe your typical morning.",
    "What matters most to you?",
]

print(f"Model: {MODEL_NAME}")
print(f"Target layer: {TARGET_LAYER}")
print(f"Roles: {list(ROLE_PROMPTS.keys())} + default")
print(f"Questions: {len(QUESTIONS)}")
print(f"Total responses to generate: {(len(ROLE_PROMPTS) + 1) * len(QUESTIONS)} "
      f"({len(ROLE_PROMPTS)} roles + 1 default) x {len(QUESTIONS)} questions")

---
## 2. Load Model

We use `ProbingModel` which wraps HuggingFace model + tokenizer. No vLLM needed.

In [ ]:
from assistant_axis.internals import ProbingModel

print(f"Loading {MODEL_NAME}...")
pm = ProbingModel(MODEL_NAME, dtype=torch.bfloat16)

n_layers = len(pm.get_layers())
print(f"Loaded! {n_layers} layers, hidden_size={pm.hidden_size}")
print(f"Using target layer {TARGET_LAYER} of {n_layers}")

# Sanity check
assert TARGET_LAYER < n_layers, f"Target layer {TARGET_LAYER} >= {n_layers} total layers"

---
## 3. Generate Responses (Pipeline Step 1 — simplified)

Instead of vLLM batch inference across 277 roles, we use HuggingFace `generate_response()` for a handful of prompts.

In [ ]:
from assistant_axis.generation import generate_response, format_conversation

all_responses = {}  # {"role_name": [{"conversation": [...], "question": ...}, ...]}

# ── Generate role-playing responses ──
for role_name, system_prompt in ROLE_PROMPTS.items():
    all_responses[role_name] = []
    print(f"\n{'='*60}")
    print(f"Generating for role: {role_name}")
    print(f"System prompt: {system_prompt[:80]}...")
    
    for q in QUESTIONS:
        conversation = format_conversation(system_prompt, q, pm.tokenizer)
        response = generate_response(
            pm.model, pm.tokenizer, conversation,
            max_new_tokens=200, temperature=0.7,
        )
        
        # Store the full conversation (including the assistant's response)
        full_conv = conversation + [{"role": "assistant", "content": response}]
        all_responses[role_name].append({
            "conversation": full_conv,
            "question": q,
            "response": response,
        })
        print(f"  Q: {q[:50]}...")
        print(f"  A: {response[:100]}...")

# ── Generate default (no persona) responses ──
all_responses["default"] = []
print(f"\n{'='*60}")
print(f"Generating for: default (no system prompt)")

for q in QUESTIONS:
    conversation = format_conversation(None, q, pm.tokenizer)
    response = generate_response(
        pm.model, pm.tokenizer, conversation,
        max_new_tokens=200, temperature=0.7,
    )
    full_conv = conversation + [{"role": "assistant", "content": response}]
    all_responses["default"].append({
        "conversation": full_conv,
        "question": q,
        "response": response,
    })
    print(f"  Q: {q[:50]}...")
    print(f"  A: {response[:100]}...")

print(f"\nDone! Generated {sum(len(v) for v in all_responses.values())} total responses.")

---
## 4. Extract Activations (Pipeline Step 2 — simplified)

For each response, run it through the model with hooks to capture the hidden state, then compute the mean activation across assistant tokens.

In [ ]:
from assistant_axis.internals import ConversationEncoder, ActivationExtractor
from assistant_axis.internals.spans import SpanMapper

encoder = ConversationEncoder(pm.tokenizer, pm.model_name)
extractor = ActivationExtractor(pm, encoder)

all_activations = {}  # {"role_name": [tensor, tensor, ...]}  each tensor is (n_layers, hidden_dim)

for role_name, responses in all_responses.items():
    all_activations[role_name] = []
    print(f"\nExtracting activations for: {role_name}")
    
    for i, resp in enumerate(responses):
        conversation = resp["conversation"]
        
        # Extract activations for the full conversation at all layers
        # Shape: (n_layers, n_tokens, hidden_dim)
        full_acts = extractor.full_conversation(conversation, layer=None)
        
        # Find which tokens belong to the assistant's response
        indices = encoder.response_indices(conversation, per_turn=False)
        
        if indices:
            # Mean of assistant tokens at each layer → (n_layers, hidden_dim)
            assistant_acts = full_acts[:, indices, :]     # (n_layers, n_assistant_tokens, hidden_dim)
            mean_act = assistant_acts.mean(dim=1)          # (n_layers, hidden_dim)
            all_activations[role_name].append(mean_act)
            print(f"  Response {i}: {len(indices)} assistant tokens → mean activation {mean_act.shape}")
        else:
            print(f"  Response {i}: WARNING — no assistant tokens found, skipping")

print(f"\nActivations extracted:")
for role_name, acts in all_activations.items():
    print(f"  {role_name}: {len(acts)} samples")

---
## 5. Assign Scores (Pipeline Step 3 — simplified)

The real pipeline uses GPT-4 to score each response 0-3. For this POC, we **skip the judge** and manually assign scores:
- Role-playing responses → score 3 (assume the model played the role)
- Default responses → no score needed (we use all of them)

This is a valid shortcut because with strong system prompts and a capable model, most responses *will* play the role. The judge exists to filter out the ~10-20% that don't — for a POC that doesn't matter.

In [ ]:
# In the real pipeline, GPT-4 assigns these scores.
# Here we assume all role-playing responses fully played the role (score=3).

scores = {}
for role_name in ROLE_PROMPTS:
    scores[role_name] = [3] * len(all_activations[role_name])

print("Scores assigned (all role responses = 3, default = N/A):")
for role_name, s in scores.items():
    print(f"  {role_name}: {s}")

---
## 6. Compute Per-Role Vectors (Pipeline Step 4)

For each role: mean of score=3 activations → one vector per role.
For default: mean of ALL activations.

In [ ]:
role_vectors = {}  # {"role_name": tensor of shape (n_layers, hidden_dim)}

# Role vectors: mean of score=3 activations
for role_name in ROLE_PROMPTS:
    acts = all_activations[role_name]
    role_scores = scores[role_name]
    
    # Filter by score == 3
    filtered = [a for a, s in zip(acts, role_scores) if s == 3]
    
    if filtered:
        stacked = torch.stack(filtered)         # (n_samples, n_layers, hidden_dim)
        role_vectors[role_name] = stacked.mean(dim=0)  # (n_layers, hidden_dim)
        print(f"  {role_name}: {len(filtered)} score=3 samples → vector {role_vectors[role_name].shape}")
    else:
        print(f"  {role_name}: WARNING — no score=3 samples!")

# Default vector: mean of ALL activations (no score filtering)
default_acts = all_activations["default"]
if default_acts:
    stacked = torch.stack(default_acts)
    role_vectors["default"] = stacked.mean(dim=0)
    print(f"  default: {len(default_acts)} samples → vector {role_vectors['default'].shape}")

print(f"\nComputed {len(role_vectors)} per-role vectors.")

---
## 7. Compute the Axis (Pipeline Step 5)

This is the moment of truth. The axis is:
```
axis = mean(default_vectors) - mean(role_vectors)
```

In [ ]:
from assistant_axis.axis import compute_axis

# Separate default and role vectors
default_vecs = [role_vectors["default"]]  # Just one default in this POC
role_vecs = [v for name, v in role_vectors.items() if name != "default"]

print(f"Default vectors: {len(default_vecs)}")
print(f"Role vectors: {len(role_vecs)} ({[n for n in role_vectors if n != 'default']})")

# Stack into (n_samples, n_layers, hidden_dim) format
default_tensor = torch.stack(default_vecs)  # (1, n_layers, hidden_dim)
role_tensor = torch.stack(role_vecs)          # (3, n_layers, hidden_dim)

# Compute axis
axis = compute_axis(role_tensor, default_tensor)

print(f"\nAxis shape: {axis.shape}")
print(f"Axis norm at target layer {TARGET_LAYER}: {axis[TARGET_LAYER].norm():.4f}")
print(f"Axis norms across layers:")
norms = axis.norm(dim=1)
for i in range(0, n_layers, max(1, n_layers // 8)):
    marker = " ← target" if i == TARGET_LAYER else ""
    print(f"  Layer {i:2d}: {norms[i]:.4f}{marker}")

---
## 8. Validate — Does the Axis Actually Work?

The acid test: project role-playing activations and default activations onto the axis. 

**If the axis works:**
- Default responses → **positive** projections (toward the "assistant" end)
- Role-playing responses → **negative** projections (toward the "role-playing" end)
- There should be clear separation between the two groups

In [ ]:
from assistant_axis.axis import project

print(f"Projections at layer {TARGET_LAYER}:")
print(f"{'='*60}")

all_role_projections = []
all_default_projections = []

# Project default activations
print(f"\nDEFAULT (should be POSITIVE):")
for i, act in enumerate(all_activations["default"]):
    proj = project(act, axis, layer=TARGET_LAYER)
    all_default_projections.append(proj)
    q = all_responses["default"][i]["question"][:40]
    print(f"  {proj:+.4f}  ← \"{q}...\"")

# Project role-playing activations
for role_name in ROLE_PROMPTS:
    print(f"\n{role_name.upper()} (should be NEGATIVE):")
    for i, act in enumerate(all_activations[role_name]):
        proj = project(act, axis, layer=TARGET_LAYER)
        all_role_projections.append(proj)
        q = all_responses[role_name][i]["question"][:40]
        print(f"  {proj:+.4f}  ← \"{q}...\"")

# Summary statistics
mean_default = sum(all_default_projections) / len(all_default_projections)
mean_role = sum(all_role_projections) / len(all_role_projections)

print(f"\n{'='*60}")
print(f"SUMMARY:")
print(f"  Default mean projection:  {mean_default:+.4f}")
print(f"  Role-playing mean projection: {mean_role:+.4f}")
print(f"  Separation (default - role):  {mean_default - mean_role:.4f}")

if mean_default > mean_role:
    print(f"\n  *** AXIS WORKS! Default projections are higher than role projections. ***")
else:
    print(f"\n  *** Unexpected: role projections are higher. This can happen with very")
    print(f"      few samples. Try with more questions or a larger model. ***")

---
## 9. Bonus — Test Activation Steering

Now let's use the axis to actually steer the model's behavior. We'll give it a role-playing prompt and use steering to push it back toward assistant behavior.

In [ ]:
from assistant_axis import ActivationSteering

test_prompt = "You are a pirate captain. Tell me about yourself."

# ── Without steering ──
print("WITHOUT STEERING:")
print("-" * 40)
unsteered = pm.generate(test_prompt, max_new_tokens=150)
print(unsteered)

# ── With steering (push toward assistant) ──
# Try a few coefficient values — higher = stronger push toward assistant behavior
for coeff in [10.0, 25.0, 50.0]:
    print(f"\nWITH STEERING (coeff={coeff}):")
    print("-" * 40)
    
    with ActivationSteering(
        pm.model,
        steering_vectors=[axis[TARGET_LAYER]],
        coefficients=[coeff],
        layer_indices=[TARGET_LAYER],
        intervention_type="addition",
    ):
        steered = pm.generate(test_prompt, max_new_tokens=150)
    print(steered)

---
## 10. Bonus — Quick PCA Check

With only 3 roles the PCA won't be statistically meaningful, but we can verify the code runs.

In [ ]:
from assistant_axis import compute_pca, MeanScaler
from assistant_axis.axis import cosine_similarity_per_layer

# Stack all per-role vectors (excluding default)
pca_input = torch.stack([v for name, v in role_vectors.items() if name != "default"])
print(f"PCA input shape: {pca_input.shape}  (n_roles, n_layers, hidden_dim)")

# Run PCA at target layer
pca_result, variance, n_components, pca_model, scaler = compute_pca(
    pca_input, layer=TARGET_LAYER, scaler=MeanScaler(), verbose=True
)

# Check: how aligned is PC1 with our computed axis?
pc1 = torch.tensor(pca_model.components_[0])  # First principal component
axis_at_layer = axis[TARGET_LAYER].float()
cos_sim = torch.nn.functional.cosine_similarity(pc1.unsqueeze(0), axis_at_layer.unsqueeze(0))
print(f"\nCosine similarity between axis and PC1: {cos_sim.item():.4f}")
print(f"(Values near +1 or -1 mean the axis aligns with the dominant direction)")

---
## 11. Save the Axis

Save it so you can reuse it without re-running the pipeline.

In [ ]:
from assistant_axis.axis import save_axis

output_path = "poc_axis.pt"
save_axis(axis, output_path, metadata={
    "model": MODEL_NAME,
    "target_layer": TARGET_LAYER,
    "n_roles": len(ROLE_PROMPTS),
    "n_questions": len(QUESTIONS),
    "roles": list(ROLE_PROMPTS.keys()),
    "note": "POC axis — not production quality (too few samples)",
})

print(f"Saved axis to {output_path}")
print(f"File size: {Path(output_path).stat().st_size / 1024:.1f} KB")

---
## 12. Cleanup

In [ ]:
pm.close()
print("Model unloaded, GPU memory freed.")

---
## What This POC Proved

If everything above ran successfully, you've validated:

| Pipeline Step | Real Pipeline | This POC |
|--------------|--------------|----------|
| 1. Generate | 277 roles x 240 questions x 5 prompts = 66K responses via vLLM | 4 roles x 5 questions = 20 responses via HuggingFace |
| 2. Activations | Batch extraction with multi-GPU | Sequential, single GPU |
| 3. Judge | GPT-4 scoring via OpenAI API | Manual scores (assumed 3) |
| 4. Vectors | Per-role mean of score=3 activations | Same logic, fewer samples |
| 5. Axis | `mean(default) - mean(role)` | Same formula |
| Projection | Verify separation | Same |
| Steering | Modify behavior via hooks | Same |

**To scale up from here:** Increase `QUESTIONS`, add more roles from `data/roles/instructions/*.json`, switch to a larger model, and use the real judge (step 3) for accurate scoring.

**Key limitation of this POC:** With only 5 samples per category, the axis is noisy. The real pipeline uses 1000+ samples per role to get a stable direction. If projections look noisy or the separation is small, that's expected — more data fixes it.